# Book Liquidity Concentration and Imbalance

This notebook studies how liquidity is distributed across the top `L` levels of the book.

The notebook has one combined diagnostic view:

- midprice context
- ask-side concentration
- bid-side concentration
- cumulative bid/ask imbalance
- bid/ask center-of-mass gap

The goal is to answer three questions:

> is the resting liquidity clustered near the touch or pushed deeper into the book?
>
> is the selected depth bid-heavy or ask-heavy?
>
> how far apart are the bid and ask concentration profiles?


## Metric Definitions

Let `bid_qty_i` and `ask_qty_i` be the resting sizes at level `i`, where level `1` is the best bid or best ask.

For a chosen depth `L`:

- `bid_depth_L = sum_{i=1..L} bid_qty_i`
- `ask_depth_L = sum_{i=1..L} ask_qty_i`
- `bid_share_i = bid_qty_i / bid_depth_L`
- `ask_share_i = ask_qty_i / ask_depth_L`
- `center_of_mass_bid_L = sum_{i=1..L} i * bid_share_i`
- `center_of_mass_ask_L = sum_{i=1..L} i * ask_share_i`
- `total_depth_L = bid_depth_L + ask_depth_L`
- `imbalance_L = (bid_depth_L - ask_depth_L) / total_depth_L`
- `center_of_mass_gap_L = center_of_mass_ask_L - center_of_mass_bid_L`

Interpretation:

- a center-of-mass value near `1` means liquidity is concentrated at the touch
- a center-of-mass value near `L` means liquidity is pushed deeper into the book
- a center-of-mass value near `(L + 1) / 2` means depth is spread more evenly across the selected levels
- `imbalance_L > 0` means bid depth is larger than ask depth
- `imbalance_L < 0` means ask depth is larger than bid depth
- `imbalance_L = 0` means the selected depth is balanced
- `center_of_mass_gap_L > 0` means ask liquidity is deeper than bid liquidity
- `center_of_mass_gap_L < 0` means bid liquidity is deeper than ask liquidity

The uniform reference `(L + 1) / 2` is shown as a dashed line on the concentration panels.
The zero reference is shown as a dashed line on the imbalance and gap panels.

The moving-average curve is just a smoothed version of the same series.
It does not change the underlying metric.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display


def find_backtester_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'stats').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root containing stats/ and notebooks/.')


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /Users/hoangdeveloper/PycharmProjects/exchange-data-backtester


In [2]:
from stats.notebook import load_orderflow_day
from stats.tables import get_or_build_book_levels_table
from stats.features.book import compute_mid_spread, resample_book

REFERENCE_DAY = '20260223'
BOOK_TOP_N = 10
PLOT_GRID_FREQ = '1s'
REPLAY_ON_GAP = 'skip-segment'

# Load one representative day and replay the top L levels from the order book.
dataset, trades, top, summary = load_orderflow_day(day=REFERENCE_DAY, replay_on_gap=REPLAY_ON_GAP)
book_levels = get_or_build_book_levels_table(
    dataset,
    top_n=BOOK_TOP_N,
    on_gap=REPLAY_ON_GAP,
    show_progress=False,
)

# Resample to a clock grid so the series is easier to inspect.
book_grid = resample_book(book_levels, grid_freq=PLOT_GRID_FREQ)
book_grid = book_grid.join(compute_mid_spread(book_grid), how='left')

print(f'book_levels rows: {len(book_levels):,}')
print(f'book_grid rows: {len(book_grid):,} at {PLOT_GRID_FREQ}')
display(summary.to_frame('value'))

book_levels rows: 722,893
book_grid rows: 80,096 at 1s


,value
exchange,binance
symbol,BTCUSDC
day,20260223
day_dir,/Users/hoangdeveloper/PycharmProjects/exchange...
trades_rows,868008
top_rows,722893
trade_start_utc,2026-02-23 01:00:04.958000+00:00
trade_end_utc,2026-02-23 23:15:00.320000+00:00
top_start_utc,2026-02-23 01:00:04.735000+00:00
top_end_utc,2026-02-23 23:14:59.835000+00:00


## Helper Functions

The notebook computes the concentration profile directly from the replayed book levels.

The main outputs are:

- `center_of_mass_bid_L`
- `center_of_mass_ask_L`
- `center_of_mass_gap_L`
- `center_of_mass_bid_ma_L`
- `center_of_mass_ask_ma_L`
- `center_of_mass_gap_ma_L`

The per-level shares are an internal normalization step used to compute the center of mass.

In [3]:
def compute_center_of_mass(frame: pd.DataFrame, *, side: str, levels: int) -> pd.Series:
    """Return the center of mass for bid or ask depth across the top `levels` levels."""
    if side not in {'bid', 'ask'}:
        raise ValueError("side must be 'bid' or 'ask'")

    qty_cols = [f'{side}{k}_qty' for k in range(1, levels + 1)]
    missing = [name for name in qty_cols if name not in frame.columns]
    if missing:
        raise KeyError(f'Missing {side} depth columns for levels={levels}: {missing[:8]}')

    depth = frame[qty_cols].sum(axis=1)
    denom = depth.replace(0, np.nan)
    shares = frame[qty_cols].div(denom, axis=0)
    weights = np.arange(1, levels + 1, dtype=float)
    return shares.mul(weights, axis=1).sum(axis=1).rename(f'center_of_mass_{side}_{levels}')


def summarize_center_of_mass(frame: pd.DataFrame, *, side: str, levels: int) -> pd.Series:
    series = compute_center_of_mass(frame, side=side, levels=levels).dropna()
    reference = (levels + 1) / 2.0
    if series.empty:
        return pd.Series(
            {
                'side': side,
                'levels': levels,
                'uniform_reference': reference,
                'observations': 0,
                'mean': np.nan,
                'median': np.nan,
                'std': np.nan,
                'p10': np.nan,
                'p90': np.nan,
                'share_below_reference': np.nan,
                'share_above_reference': np.nan,
            }
        )

    return pd.Series(
        {
            'side': side,
            'levels': levels,
            'uniform_reference': reference,
            'observations': int(series.size),
            'mean': float(series.mean()),
            'median': float(series.median()),
            'std': float(series.std()),
            'p10': float(series.quantile(0.10)),
            'p90': float(series.quantile(0.90)),
            'share_below_reference': float((series < reference).mean()),
            'share_above_reference': float((series > reference).mean()),
        }
    )


def compute_book_imbalance(frame: pd.DataFrame, *, levels: int) -> pd.DataFrame:
    bid_cols = [f'bid{k}_qty' for k in range(1, levels + 1)]
    ask_cols = [f'ask{k}_qty' for k in range(1, levels + 1)]
    missing = [name for name in bid_cols + ask_cols if name not in frame.columns]
    if missing:
        raise KeyError(f'Missing depth columns for levels={levels}: {missing[:8]}')

    bid_depth = frame[bid_cols].sum(axis=1)
    ask_depth = frame[ask_cols].sum(axis=1)
    total_depth = bid_depth + ask_depth
    denom = total_depth.replace(0, np.nan)

    out = pd.DataFrame(index=frame.index)
    out[f'bid_depth_{levels}'] = bid_depth
    out[f'ask_depth_{levels}'] = ask_depth
    out[f'total_depth_{levels}'] = total_depth
    out[f'imbalance_{levels}'] = (bid_depth - ask_depth) / denom
    out[f'share_bid_{levels}'] = bid_depth / denom
    return out


def summarize_book_imbalance(frame: pd.DataFrame, *, levels: int) -> pd.Series:
    feat = compute_book_imbalance(frame, levels=levels)
    series = feat[f'imbalance_{levels}'].dropna()
    if series.empty:
        return pd.Series(
            {
                'levels': levels,
                'observations': 0,
                'mean_imbalance': np.nan,
                'median_imbalance': np.nan,
                'std_imbalance': np.nan,
                'p10_imbalance': np.nan,
                'p90_imbalance': np.nan,
                'share_positive': np.nan,
                'mean_total_depth': np.nan,
                'mean_bid_depth': np.nan,
                'mean_ask_depth': np.nan,
            }
        )

    return pd.Series(
        {
            'levels': levels,
            'observations': int(series.size),
            'mean_imbalance': float(series.mean()),
            'median_imbalance': float(series.median()),
            'std_imbalance': float(series.std()),
            'p10_imbalance': float(series.quantile(0.10)),
            'p90_imbalance': float(series.quantile(0.90)),
            'share_positive': float((series > 0).mean()),
            'mean_total_depth': float(feat[f'total_depth_{levels}'].mean()),
            'mean_bid_depth': float(feat[f'bid_depth_{levels}'].mean()),
            'mean_ask_depth': float(feat[f'ask_depth_{levels}'].mean()),
        }
    )


def smooth_clock_series(series: pd.Series, *, window_min: int, min_periods: int | None = None) -> pd.Series:
    window = pd.Timedelta(minutes=int(window_min))
    if min_periods is None:
        min_periods = max(5, int(window_min) * 5)
    return series.rolling(window, min_periods=min_periods).mean()


def smooth_center_of_mass(series: pd.Series, *, window_min: int, min_periods: int | None = None) -> pd.Series:
    return smooth_clock_series(series, window_min=window_min, min_periods=min_periods)


def compute_center_of_mass_gap(frame: pd.DataFrame, *, levels: int) -> pd.Series:
    bid = compute_center_of_mass(frame, side='bid', levels=levels)
    ask = compute_center_of_mass(frame, side='ask', levels=levels)
    return (ask - bid).rename(f'center_of_mass_gap_{levels}')


def summarize_center_of_mass_gap(frame: pd.DataFrame, *, levels: int) -> pd.Series:
    series = compute_center_of_mass_gap(frame, levels=levels).dropna()
    if series.empty:
        return pd.Series(
            {
                'levels': levels,
                'observations': 0,
                'mean_gap': np.nan,
                'median_gap': np.nan,
                'std_gap': np.nan,
                'p10_gap': np.nan,
                'p90_gap': np.nan,
                'share_positive': np.nan,
                'share_negative': np.nan,
            }
        )

    return pd.Series(
        {
            'levels': levels,
            'observations': int(series.size),
            'mean_gap': float(series.mean()),
            'median_gap': float(series.median()),
            'std_gap': float(series.std()),
            'p10_gap': float(series.quantile(0.10)),
            'p90_gap': float(series.quantile(0.90)),
            'share_positive': float((series > 0).mean()),
            'share_negative': float((series < 0).mean()),
        }
    )


In [4]:
candidate_levels = sorted({1, 3, 5, BOOK_TOP_N})
summary_table = pd.DataFrame(
    [
        summarize_center_of_mass(book_grid, side=side, levels=level)
        for side in ('bid', 'ask')
        for level in candidate_levels
    ]
)

summary_table = summary_table[
    [
        'side',
        'levels',
        'uniform_reference',
        'observations',
        'mean',
        'median',
        'std',
        'p10',
        'p90',
        'share_below_reference',
        'share_above_reference',
    ]
]

with pd.option_context('display.float_format', '{:.4f}'.format):
    display(summary_table)

imbalance_summary = pd.DataFrame(
    [summarize_book_imbalance(book_grid, levels=level) for level in candidate_levels]
)

imbalance_summary = imbalance_summary[
    [
        'levels',
        'observations',
        'mean_imbalance',
        'median_imbalance',
        'std_imbalance',
        'p10_imbalance',
        'p90_imbalance',
        'share_positive',
        'mean_total_depth',
        'mean_bid_depth',
        'mean_ask_depth',
    ]
]

with pd.option_context('display.float_format', '{:.4f}'.format):
    display(imbalance_summary)

# Gap summary is kept next to the same diagnostic family.
gap_summary = pd.DataFrame(
    [summarize_center_of_mass_gap(book_grid, levels=level) for level in candidate_levels]
)

gap_summary = gap_summary[
    [
        'levels',
        'observations',
        'mean_gap',
        'median_gap',
        'std_gap',
        'p10_gap',
        'p90_gap',
        'share_positive',
        'share_negative',
    ]
]

with pd.option_context('display.float_format', '{:.4f}'.format):
    display(gap_summary)


,side,levels,uniform_reference,observations,mean,median,std,p10,p90,share_below_reference,share_above_reference
0,bid,1,1.0000,80096,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000,0.0000
1,bid,3,2.0000,80096,1.3375,1.2409,0.3599,1.0033,1.7953,0.9381,0.0561
2,bid,5,3.0000,80096,2.0317,1.8560,0.7052,1.3318,3.0000,0.8988,0.1001
3,bid,10,5.5000,80096,4.3225,4.1106,1.4876,2.5798,6.3881,0.7916,0.2084
4,ask,1,1.0000,80096,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000,0.0000
5,ask,3,2.0000,80096,1.3323,1.2471,0.3513,1.0031,1.7758,0.9398,0.0555
6,ask,5,3.0000,80096,2.0326,1.8674,0.6781,1.3552,2.9903,0.9029,0.0964
7,ask,10,5.5000,80096,4.3692,4.1874,1.4524,2.6417,6.3635,0.7820,0.2180


,levels,observations,mean_imbalance,median_imbalance,std_imbalance,p10_imbalance,p90_imbalance,share_positive,mean_total_depth,mean_bid_depth,mean_ask_depth
0,1.0000,80096.0000,0.0142,0.0196,0.5844,-0.8114,0.8261,0.5123,0.6385,0.3263,0.3122
1,3.0000,80096.0000,0.0147,0.0226,0.5677,-0.7758,0.7963,0.5128,0.7972,0.4065,0.3907
2,5.0000,80096.0000,0.0142,0.0210,0.5358,-0.7295,0.7463,0.5129,1.0099,0.5182,0.4917
3,10.0000,80096.0000,0.0115,0.0139,0.5009,-0.6765,0.7002,0.5081,1.7011,0.8828,0.8183


,levels,observations,mean_gap,median_gap,std_gap,p10_gap,p90_gap,share_positive,share_negative
0,1.0000,80096.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,3.0000,80096.0000,-0.0052,-0.0014,0.4793,-0.5258,0.5094,0.4915,0.5085
2,5.0000,80096.0000,0.0009,0.0129,0.9544,-1.1754,1.1537,0.5074,0.4926
3,10.0000,80096.0000,0.0467,0.0669,2.1227,-2.7003,2.7764,0.5141,0.4859


## Interactive Concentration Plot

Use the sliders to choose:

- `L levels`: how many book levels to include
- `MA min`: the smoothing window used for the moving-average curve
- `imb th`: the imbalance threshold used to mark buy/sell states on the price plot. Rolling mode usually needs a lower value because smoothing suppresses extremes.
- `imb mode`: whether the markers use the instantaneous imbalance or the rolling imbalance

The plot shows:

- midprice for context
- buy and sell imbalance markers on the price panel
- ask center of mass and its moving average
- bid center of mass and its moving average
- cumulative bid/ask imbalance and its moving average
- the bid/ask center-of-mass gap and its moving average

The dashed horizontal lines are the uniform reference `(L + 1) / 2` for the concentration panels and `0` for the imbalance and gap panels.


In [5]:
def make_concentration_figure(levels: int, ma_min: int, imb_th: float, imb_mode: str) -> go.Figure:
    bid_raw = compute_center_of_mass(book_grid, side='bid', levels=levels)
    ask_raw = compute_center_of_mass(book_grid, side='ask', levels=levels)
    bid_ma = smooth_clock_series(bid_raw, window_min=ma_min)
    ask_ma = smooth_clock_series(ask_raw, window_min=ma_min)
    imbalance_raw = compute_book_imbalance(book_grid, levels=levels)[f'imbalance_{levels}']
    imbalance_ma = smooth_clock_series(imbalance_raw, window_min=ma_min)
    gap_raw = compute_center_of_mass_gap(book_grid, levels=levels)
    gap_ma = smooth_clock_series(gap_raw, window_min=ma_min)
    reference = (levels + 1) / 2.0

    fig = make_subplots(
        rows=5,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.18, 0.18, 0.18, 0.22, 0.24],
        subplot_titles=(
            'Midprice context',
            f'Ask center of mass over the top {levels} levels',
            f'Bid center of mass over the top {levels} levels',
            f'Cumulative imbalance over the top {levels} levels',
            f'Bid/ask center of mass gap over the top {levels} levels',
        ),
    )

    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=book_grid['mid'],
            mode='lines',
            name='midprice',
            line=dict(color='#2f3640', width=1.3),
            hovertemplate='%{x}<br>mid=%{y:.2f}<extra></extra>',
        ),
        row=1,
        col=1,
    )

    marker_signal = imbalance_raw if imb_mode == 'instant' else imbalance_ma
    marker_label = 'instant' if imb_mode == 'instant' else f'rolling MA {ma_min} min'
    buy_mask = marker_signal >= imb_th
    sell_mask = marker_signal <= -imb_th
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index[buy_mask],
            y=book_grid.loc[buy_mask, 'mid'],
            mode='markers',
            name=f'buy {marker_label} >= {imb_th:.2f}',
            marker=dict(color='#2ca02c', size=7, symbol='triangle-up', line=dict(width=0.5, color='#1b5e20')),
            hovertemplate='%{x}<br>buy imbalance marker<extra></extra>',
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index[sell_mask],
            y=book_grid.loc[sell_mask, 'mid'],
            mode='markers',
            name=f'sell {marker_label} <= -{imb_th:.2f}',
            marker=dict(color='#d62728', size=7, symbol='triangle-down', line=dict(width=0.5, color='#7f1d1d')),
            hovertemplate='%{x}<br>sell imbalance marker<extra></extra>',
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=ask_raw,
            mode='lines',
            name=f'ask raw L={levels}',
            line=dict(color='#ff7f0e', width=1.1),
            opacity=0.35,
            hovertemplate='%{x}<br>ask COM=%{y:.3f}<extra></extra>',
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=ask_ma,
            mode='lines',
            name=f'ask MA {ma_min} min',
            line=dict(color='#ff7f0e', width=2.2),
            hovertemplate='%{x}<br>ask COM MA=%{y:.3f}<extra></extra>',
        ),
        row=2,
        col=1,
    )
    fig.add_hline(
        y=reference,
        line_dash='dash',
        line_color='#d62728',
        row=2,
        col=1,
        annotation_text=f'uniform reference = {reference:.2f}',
        annotation_position='top right',
    )

    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=bid_raw,
            mode='lines',
            name=f'bid raw L={levels}',
            line=dict(color='#1f77b4', width=1.1),
            opacity=0.35,
            hovertemplate='%{x}<br>bid COM=%{y:.3f}<extra></extra>',
        ),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=bid_ma,
            mode='lines',
            name=f'bid MA {ma_min} min',
            line=dict(color='#1f77b4', width=2.2),
            hovertemplate='%{x}<br>bid COM MA=%{y:.3f}<extra></extra>',
        ),
        row=3,
        col=1,
    )
    fig.add_hline(
        y=reference,
        line_dash='dash',
        line_color='#d62728',
        row=3,
        col=1,
        annotation_text=f'uniform reference = {reference:.2f}',
        annotation_position='top right',
    )

    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=imbalance_raw,
            mode='lines',
            name=f'imbalance raw L={levels}',
            line=dict(color='#2ca02c', width=1.1),
            opacity=0.35,
            hovertemplate='%{x}<br>imbalance=%{y:.3f}<extra></extra>',
        ),
        row=4,
        col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=imbalance_ma,
            mode='lines',
            name=f'imbalance MA {ma_min} min',
            line=dict(color='#2ca02c', width=2.2),
            hovertemplate='%{x}<br>imbalance MA=%{y:.3f}<extra></extra>',
        ),
        row=4,
        col=1,
    )
    fig.add_hline(
        y=0.0,
        line_dash='dash',
        line_color='#d62728',
        row=4,
        col=1,
        annotation_text='balanced book = 0',
        annotation_position='top right',
    )

    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=gap_raw,
            mode='lines',
            name=f'gap raw L={levels}',
            line=dict(color='#9467bd', width=1.1),
            opacity=0.35,
            hovertemplate='%{x}<br>gap=%{y:.3f}<extra></extra>',
        ),
        row=5,
        col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=book_grid.index,
            y=gap_ma,
            mode='lines',
            name=f'gap MA {ma_min} min',
            line=dict(color='#9467bd', width=2.2),
            hovertemplate='%{x}<br>gap MA=%{y:.3f}<extra></extra>',
        ),
        row=5,
        col=1,
    )
    fig.add_hline(
        y=0.0,
        line_dash='dash',
        line_color='#d62728',
        row=5,
        col=1,
        annotation_text='balanced gap = 0',
        annotation_position='top right',
    )

    fig.update_yaxes(title_text='midprice', row=1, col=1)
    fig.update_yaxes(title_text=f'center_of_mass_ask_{levels}', row=2, col=1, range=[0.8, levels + 0.2])
    fig.update_yaxes(title_text=f'center_of_mass_bid_{levels}', row=3, col=1, range=[levels + 0.2, 0.8])
    fig.update_yaxes(title_text=f'imbalance_{levels}', row=4, col=1, range=[-1.05, 1.05])
    fig.update_yaxes(title_text=f'center_of_mass_gap_{levels}', row=5, col=1)
    fig.update_xaxes(title_text='UTC', row=5, col=1)
    buy_count = int(buy_mask.sum())
    sell_count = int(sell_mask.sum())

    fig.update_layout(
        template='plotly_white',
        width=1280,
        height=1500,
        hovermode='x unified',
        title=f'Book liquidity concentration and imbalance | top {levels} levels | MA {ma_min} min | th {imb_th:.2f} | mode {imb_mode} | markers {buy_count + sell_count}',
        margin=dict(l=60, r=30, t=80, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0.0),
    )
    return fig


level_slider = widgets.IntSlider(
    value=min(5, BOOK_TOP_N),
    min=1,
    max=BOOK_TOP_N,
    step=1,
    description='L levels',
    continuous_update=False,
)
ma_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=60,
    step=1,
    description='MA min',
    continuous_update=False,
)

imb_slider = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=0.95,
    step=0.05,
    description='imb th',
    continuous_update=False,
)

imb_mode = widgets.Dropdown(
    options=[('Instantaneous', 'instant'), ('Rolling MA', 'rolling')],
    value='instant',
    description='imb mode',
)

widgets.interact(make_concentration_figure, levels=level_slider, ma_min=ma_slider, imb_th=imb_slider, imb_mode=imb_mode);


interactive(children=(IntSlider(value=5, continuous_update=False, description='L levels', max=10, min=1), IntS…

## Notes

The notebook keeps the metrics explicit:

- concentration is about where liquidity sits within each side
- imbalance is about whether the selected depth is bid-heavy or ask-heavy
- the center-of-mass gap is a direct asymmetry measure between the two sides
- the moving-average curves are only smoothing aids

That gives one compact view with five aligned panels below the price graph.
